## QF620 Final Project: SPX Options During the 2025 Liberation Day Tariff Shock
Objective

```
Understand how market expectations, left-tail risk, and geopolitical retaliation
are priced dynamically, in real-time, by high-frequency SPX options and ES
futures market data.
```

In [41]:
# Initiate all libraries
import os
import datetime as dt

import numpy as np
import pandas as pd
import matplotlib
from scipy.stats import norm
from scipy.optimize import brentq
from plotnine import *
from mizani.formatters import percent_format

RISK_FREE_RATE = 0.0421  # constant risk-free rate, FRED April 2025 (continuously compounded)

## Reusable functions

In [42]:
def nearest_to_minute(df):
    """Used when pulling data to trim the sample down to the nearest minute"""
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    df = df[~df.index.duplicated(keep="last")].sort_index()
    # Only minutes that have a quote within +/-30s (skips weekends and session gaps)
    minutes = pd.DataFrame(index=df.index.round("min").unique().rename("DATETIME"))
    return pd.merge_asof(
        minutes,
        df,
        left_index=True,
        right_index=True,
        direction="nearest",
        tolerance=pd.Timedelta("30s"),
    )

## Data Prep

CSV Files
1. Asset Prices (*assets.csv) are time series for SPX, VIX, Bid and Ask for ES Futures

2. SPX Options Prices (call_bid, call_ask, put_bid, put_ask) are the time series prices for different strike price.
    1. 'ETH' or 'RTH' is stored in the column TRADING_HOUR if it is needed for further analysis
    2. Expiry date is stored in the suffix of the file. E.g. `2025-04-01_call_ask_20250416.csv` has price data of 01Apr2025 but Expiry 16Apr2025.

3. Event data taken from `2025 Liberation Day Tariff Main Events.csv` contains important happenings during the time from 01Apr to 10Apr. `General_Event_Type` column created based on slide 2 of `Project (2026-27 T1).pdf`

### EVENT

In [43]:
EVENT = pd.read_csv("2025 Liberation Day Tariff Main Events.csv", parse_dates=True)
event_time = pd.to_datetime(EVENT["Date_ET"].astype(str) + " " + EVENT["Time_ET"].astype(str))
PHASE_EDGES = pd.to_datetime([
    "2025-04-01 00:00",  # Apr 1-2
    "2025-04-03 00:00",  # Apr 3-4
    "2025-04-05 00:00",  # Apr 5-7
    "2025-04-07 09:00",  # Apr 7-9 am
    "2025-04-09 21:00",  # Apr 9 pm onward
    "2025-04-11 00:00",
])
PHASE_LABELS = [
    "Anticipation → Shock",
    "Retaliation became real",
    "Market panic did not lead to US policy retreat",
    "US-China escalation",
    "90-day negotiation window",
]
EVENT["General_Event_Type"] = pd.cut(event_time, bins=PHASE_EDGES, labels=PHASE_LABELS, right=False)

# Concatenate Date_ET and Time_ET into datetime column and set it as the index
EVENT['DATETIME'] = pd.to_datetime(EVENT['Date_ET'].astype(str) + ' ' + EVENT['Time_ET'].astype(str))
EVENT.set_index('DATETIME', inplace=True)
EVENT.sort_index(inplace=True)
EVENT

,Date_ET,Time_ET,ET_Zone,Country_or_Actor,Announcer,Event_Type,Target,Event_Description,Tariff_Rate_or_Change,Direction,General_Event_Type
DATETIME,,,,,,,,,,,
2025-04-01 12:29:00,2025-04-01,12:29,EDT,United States,White House,Policy confirmation,Global,White House confirms Apr 2 reciprocal-tariff a...,NaN,Announcement,Anticipation → Shock
2025-04-02 16:07:00,2025-04-02,16:07,EDT,United States,President Trump,Tariff announcement,Global,"""Liberation Day"": 10% baseline tariff on most ...",10% baseline; country-specific rates,Announcement,Anticipation → Shock
2025-04-03 00:01:00,2025-04-03,0:01,EDT,United States,U.S. Government,Tariff effective,Automobiles,Previously announced 25% tariff on imported au...,25%,Effective,Retaliation became real
2025-04-03 11:30:00,2025-04-03,11:30,EDT,Canada,Prime Minister Mark Carney,Retaliatory tariff announcement,United States,Canada announces 25% retaliatory tariffs on se...,25%,Retaliation,Retaliation became real
2025-04-03 22:00:00,2025-04-03,22:00,EDT,China,Ministry of Commerce,Retaliation warning,United States,China condemns U.S. reciprocal tariffs and say...,NaN,Retaliation warning,Retaliation became real
2025-04-04 06:00:00,2025-04-04,6:00,EDT,China,Ministry of Commerce,WTO dispute,United States,China says it has filed a WTO dispute against ...,NaN,Legal retaliation,Retaliation became real
2025-04-04 07:21:00,2025-04-04,7:21,EDT,China,State Council Tariff Commission / MOFCOM,Retaliatory tariff announcement,United States,China announces an additional 34% tariff on al...,34%,Retaliation,Retaliation became real
2025-04-05 00:01:00,2025-04-05,0:01,EDT,United States,U.S. Government,Tariff effective,Global,The 10% universal baseline tariff announced Ap...,10%,Effective,Market panic did not lead to US policy retreat
2025-04-06 19:26:00,2025-04-06,19:26,EDT,United States,President Trump,Policy signal,Global,Trump says markets may have to 'take medicine'...,NaN,Policy signal,Market panic did not lead to US policy retreat


### ASSET

In [44]:
frames = []
for hour in ["ETH", "RTH"]:
    for file in sorted(os.listdir(hour)):
        if file.endswith("assets.csv"):
            df = pd.read_csv(os.path.join(hour, file), index_col=0)
            df["TRADING_HOUR"] = hour
            frames.append(df)

# Trim down data to the nearest minute
ASSET = nearest_to_minute(pd.concat(frames))
# Configure the DataFrame's index
ASSET.index.name = "DATETIME"
ASSET.index = pd.to_datetime(ASSET.index)
ASSET.sort_index(inplace=True)
print(f"\nDataFrame ASSET has shape {ASSET.shape} and index range {ASSET.index.min()} to {ASSET.index.max()}")
display(ASSET)


DataFrame ASSET has shape (8632, 5) and index range 2025-04-01 09:31:00 to 2025-04-11 09:15:00


,SPX,ES_BID,ES_ASK,VIX,TRADING_HOUR
DATETIME,,,,,
2025-04-01 09:31:00,5590.32,5631.75,5632.0,22.66,RTH
2025-04-01 09:32:00,5586.75,5628.75,5629.0,22.72,RTH
2025-04-01 09:33:00,5593.29,5635.0,5635.25,22.55,RTH
2025-04-01 09:34:00,5590.86,5633.75,5634.0,22.57,RTH
2025-04-01 09:35:00,5589.55,5633.25,5633.5,22.64,RTH
...,...,...,...,...,...
2025-04-11 09:11:00,5248.8,5282.5,5283.0,44.33,ETH
2025-04-11 09:12:00,5253.05,5286.75,5287.25,44.35,ETH
2025-04-11 09:13:00,5248.3,5282.0,5282.5,44.32,ETH


### SPX Options Prices

In [45]:
# Map each file-name pattern to its dataset
PATTERNS = {
    "call_bid": "OPTION_CALL_BID",
    "call_ask": "OPTION_CALL_ASK",
    "put_bid": "OPTION_PUT_BID",
    "put_ask": "OPTION_PUT_ASK",
}

frames = {name: [] for name in PATTERNS.values()}
for hour in ["ETH", "RTH"]:
    for file in sorted(os.listdir(hour)):
        for pattern, name in PATTERNS.items():
            if pattern in file:
                df = pd.read_csv(os.path.join(hour, file), index_col=0)
                df["TRADING_HOUR"] = hour
                # Option files end in the expiry date, e.g. 2025-04-01_call_ask_20250416.csv
                expiry = file.rsplit("_", 1)[-1].removesuffix(".csv")
                df["EXPIRY_DATE"] = pd.to_datetime(expiry, format="%Y%m%d")
                frames[name].append(df)
                break

data = {name: pd.concat(dfs) for name, dfs in frames.items()}

for df_name, df in data.items():
    # Trim down data to the nearest minute
    df = nearest_to_minute(df)
    # Configure each DataFrame's index
    df.index.name = "DATETIME"
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)
    data[df_name] = df  # store the trimmed frame back, otherwise it is discarded
    print(f"\nDataFrame {df_name} has shape {df.shape} and index range {df.index.min()} to {df.index.max()}")
    display(df)

OPTION_CALL_BID = data["OPTION_CALL_BID"]
OPTION_CALL_ASK = data["OPTION_CALL_ASK"]
OPTION_PUT_BID = data["OPTION_PUT_BID"]
OPTION_PUT_ASK = data["OPTION_PUT_ASK"]


DataFrame OPTION_CALL_BID has shape (8632, 83) and index range 2025-04-01 09:31:00 to 2025-04-11 09:15:00


,4600.0,4625.0,4650.0,4675.0,4700.0,4725.0,4750.0,4775.0,4800.0,4825.0,...,6425.0,6450.0,6475.0,6500.0,6525.0,6550.0,6575.0,6600.0,TRADING_HOUR,EXPIRY_DATE
DATETIME,,,,,,,,,,,,,,,,,,,,,
2025-04-01 09:31:00,995.2,971.7,945.7,921.4,896.6,872.6,847.1,823.1,798.4,773.1,...,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH,2025-04-16
2025-04-01 09:32:00,993.1,968.6,943.5,918.8,894.0,869.3,844.5,819.8,795.1,770.4,...,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH,2025-04-16
2025-04-01 09:33:00,1000.5,975.8,951.0,926.2,901.5,876.7,852.0,827.3,802.6,777.9,...,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH,2025-04-16
2025-04-01 09:34:00,998.4,973.6,948.8,924.0,899.3,874.6,849.8,825.1,800.4,775.7,...,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH,2025-04-16
2025-04-01 09:35:00,996.9,971.1,946.4,921.9,896.8,872.4,847.4,822.7,797.9,773.3,...,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH,2025-04-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-11 09:11:00,737.9,716.2,694.7,673.3,651.8,630.9,610.1,589.4,568.6,548.3,...,0.35,0.3,0.3,0.25,0.25,0.2,0.2,0.2,ETH,2025-05-15
2025-04-11 09:12:00,735.0,713.2,691.7,670.2,648.9,627.7,607.2,586.0,565.3,544.9,...,0.2,0.15,0.15,0.1,0.1,0.1,0.05,0.1,ETH,2025-05-15
2025-04-11 09:13:00,737.9,716.2,694.7,673.3,652.1,631.1,610.2,589.5,569.0,548.7,...,0.15,0.15,0.15,0.1,0.1,0.1,0.05,0.1,ETH,2025-05-15



DataFrame OPTION_CALL_ASK has shape (8632, 83) and index range 2025-04-01 09:31:00 to 2025-04-11 09:15:00


,4600.0,4625.0,4650.0,4675.0,4700.0,4725.0,4750.0,4775.0,4800.0,4825.0,...,6425.0,6450.0,6475.0,6500.0,6525.0,6550.0,6575.0,6600.0,TRADING_HOUR,EXPIRY_DATE
DATETIME,,,,,,,,,,,,,,,,,,,,,
2025-04-01 09:31:00,1006.3,981.5,956.7,932.0,907.2,882.5,857.7,833.0,808.3,783.6,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,RTH,2025-04-16
2025-04-01 09:32:00,1004.7,979.7,955.2,930.4,905.7,880.9,856.2,831.5,806.8,782.1,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,RTH,2025-04-16
2025-04-01 09:33:00,1011.7,985.9,961.0,936.3,911.7,886.8,862.2,837.4,812.6,788.0,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,RTH,2025-04-16
2025-04-01 09:34:00,1009.8,985.0,959.7,935.5,910.7,885.9,861.2,836.5,811.7,787.0,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,RTH,2025-04-16
2025-04-01 09:35:00,1006.5,981.8,957.1,932.4,907.5,882.7,858.1,833.2,808.5,783.8,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,RTH,2025-04-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-11 09:11:00,742.3,720.6,699.1,677.7,656.2,635.3,614.5,593.7,572.9,552.6,...,0.95,0.9,0.85,0.85,0.8,0.8,0.75,0.75,ETH,2025-05-15
2025-04-11 09:12:00,752.7,730.9,709.7,688.1,666.7,643.0,624.9,604.0,583.4,563.0,...,0.95,0.9,0.85,0.85,0.8,0.8,0.75,0.75,ETH,2025-05-15
2025-04-11 09:13:00,742.3,720.6,699.1,677.7,656.5,635.5,614.6,593.9,573.4,553.0,...,0.95,0.9,0.85,0.85,0.8,0.75,0.75,0.75,ETH,2025-05-15



DataFrame OPTION_PUT_BID has shape (8632, 83) and index range 2025-04-01 09:31:00 to 2025-04-11 09:15:00


,4600.0,4625.0,4650.0,4675.0,4700.0,4725.0,4750.0,4775.0,4800.0,4825.0,...,6425.0,6450.0,6475.0,6500.0,6525.0,6550.0,6575.0,6600.0,TRADING_HOUR,EXPIRY_DATE
DATETIME,,,,,,,,,,,,,,,,,,,,,
2025-04-01 09:31:00,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,814.6,839.0,863.9,888.9,913.8,938.7,963.7,988.6,RTH,2025-04-16
2025-04-01 09:32:00,3.2,3.3,3.5,3.7,3.9,4.0,4.2,4.4,4.6,4.9,...,816.8,841.8,866.6,891.7,916.6,941.5,966.5,991.4,RTH,2025-04-16
2025-04-01 09:33:00,3.1,3.2,3.4,3.6,3.8,4.0,4.2,4.4,4.6,4.9,...,811.7,838.4,861.6,888.2,911.5,936.4,961.4,988.0,RTH,2025-04-16
2025-04-01 09:34:00,3.1,3.2,3.4,3.5,3.7,3.9,4.1,4.3,4.5,4.7,...,811.1,836.0,861.0,885.9,910.9,935.8,960.7,985.7,RTH,2025-04-16
2025-04-01 09:35:00,3.1,3.2,3.4,3.6,3.7,3.9,4.1,4.3,4.5,4.8,...,812.3,837.2,862.1,887.1,912.0,937.3,962.2,986.8,RTH,2025-04-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-11 09:11:00,75.0,78.1,81.5,85.0,88.6,92.4,96.4,100.5,105.0,109.4,...,1145.0,1169.6,1194.7,1219.5,1244.5,1269.3,1294.2,1319.1,ETH,2025-05-15
2025-04-11 09:12:00,74.4,77.5,80.8,84.3,87.9,91.6,95.6,99.7,104.0,108.4,...,1144.1,1165.1,1190.0,1214.9,1241.7,1264.6,1291.5,1314.3,ETH,2025-05-15
2025-04-11 09:13:00,75.4,78.7,82.0,85.5,89.2,93.0,97.0,100.9,105.5,109.8,...,1145.1,1170.0,1194.9,1219.7,1244.0,1269.1,1293.7,1318.6,ETH,2025-05-15



DataFrame OPTION_PUT_ASK has shape (8632, 83) and index range 2025-04-01 09:31:00 to 2025-04-11 09:15:00


,4600.0,4625.0,4650.0,4675.0,4700.0,4725.0,4750.0,4775.0,4800.0,4825.0,...,6425.0,6450.0,6475.0,6500.0,6525.0,6550.0,6575.0,6600.0,TRADING_HOUR,EXPIRY_DATE
DATETIME,,,,,,,,,,,,,,,,,,,,,
2025-04-01 09:31:00,3.5,3.6,3.8,4.0,4.1,4.3,4.5,4.7,5.0,5.2,...,832.7,857.5,882.6,903.5,932.3,957.4,982.3,1003.2,RTH,2025-04-16
2025-04-01 09:32:00,3.5,3.7,3.9,4.0,4.2,4.4,4.6,4.8,5.0,5.3,...,834.9,859.9,884.7,909.8,934.7,959.6,984.6,1009.5,RTH,2025-04-16
2025-04-01 09:33:00,3.4,3.6,3.7,3.9,4.1,4.3,4.5,4.7,4.9,5.2,...,829.9,854.8,879.7,904.7,929.6,954.5,979.5,1004.4,RTH,2025-04-16
2025-04-01 09:34:00,3.4,3.6,3.8,3.9,4.1,4.3,4.5,4.7,4.9,5.2,...,826.8,852.3,876.7,902.4,926.6,951.7,976.5,1002.2,RTH,2025-04-16
2025-04-01 09:35:00,3.5,3.6,3.8,4.0,4.1,4.3,4.5,4.7,5.0,5.2,...,826.8,852.3,876.7,902.4,926.6,951.7,976.5,1002.2,RTH,2025-04-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-11 09:11:00,78.1,81.3,84.7,88.0,91.7,95.7,99.6,103.8,108.2,112.8,...,1163.7,1188.6,1213.5,1238.3,1263.2,1288.1,1312.9,1337.0,ETH,2025-05-15
2025-04-11 09:12:00,77.0,80.2,83.5,87.0,90.7,94.5,98.5,102.6,107.0,111.5,...,1159.0,1183.8,1208.7,1233.6,1258.4,1283.3,1308.1,1333.0,ETH,2025-05-15
2025-04-11 09:13:00,78.1,81.3,84.7,88.3,92.0,95.7,99.9,104.1,108.5,113.1,...,1163.8,1188.7,1213.6,1238.4,1263.3,1288.2,1313.0,1337.0,ETH,2025-05-15


## 1. Plot volatility smile
Convert the option prices at one minute into an implied volatility for each strike. Following Session 6, the implied volatility at strike $K$ is the $\sigma_K$ that, put into the pricing formula, reproduces the observed market price. There is one $\sigma$ per strike, and plotting $\sigma_K$ against $K$ gives the smile. The calculation follows the course reference `implied_volatility.ipynb`, which needs only $S$, $K$, $r$, the option price and $T$.

**Assumptions**
- **Risk-free rate:** constant $r = 4.21\%$ (FRED, April 2025), continuously compounded. Stored as `RISK_FREE_RATE` and used throughout the project.
- **Underlying price $S$:** the SPX index level in `ASSET` at the same minute (section 1.2). No dividend yield.
- **Option price:** mid price $= (\text{bid} + \text{ask}) / 2$. A quote of `-1` in the files means no quote and is treated as missing. A strike whose bid is $0$ has no reliable mid and is dropped.
- **Which option per strike:** only out-of-the-money (OTM) options, since these are the liquid ones (Session 8: "low-strike puts and high-strike calls"). Puts for $K < F$, calls for $K \ge F$, where $F = S e^{rT}$ (section 1.2).

**Code structure:** each helper function below is self-contained and does not call another helper. `iv_smile` is the only function that calls the others.

### 1.1 Time to expiry
SPXW options are PM-settled on the 16:00 ET close of the expiry date. Time to expiry in years (Actual/365):

$$T = \frac{(\text{expiry date} + 16{:}00) - t}{365 \text{ days}}$$

In [46]:
def time_to_expiry(time_snapshot, expiry_date):
    """Time to expiry in years from the snapshot to 16:00 ET on the expiry date (Actual/365)."""
    expiry_time = pd.Timestamp(expiry_date) + pd.Timedelta(hours=16)
    return (expiry_time - pd.Timestamp(time_snapshot)) / pd.Timedelta(days=365)

### 1.2 Underlying price $S$ and the at-the-money point $F$
**Underlying price.** $S$ is the SPX index level in `ASSET` at the same minute, except in one window described below.

**Known data issue and workaround (6 April 20:15 to 7 April 09:30).** The SPX index is only calculated during regular trading hours, so outside them the `SPX` column is an estimate from the data vendor that normally moves with ES at 25–45 points below it. From the Sunday 6 April 20:15 open of the overnight session until the 7 April 09:30 open, that estimate sits a constant 286 points *above* ES, at about 5,180, while the option prices put the market near 4,880. A forward built from it is about 300 points too high, which breaks the 6 April smile. For minutes in this window only, the ES mid price is used as the forward and discounted to an equivalent spot level:

$$F = \frac{\text{ES\_BID} + \text{ES\_ASK}}{2}, \qquad S = F\, e^{-rT}$$

Everywhere else, $S$ is the SPX value and ES is not used.

**At-the-money point.** To decide which option is out of the money at each strike, we compare $K$ with the forward price for the option's expiry. Under the risk-neutral measure the index grows at the risk-free rate (Session 2, $\mathbb{E}^*[S_T] = S e^{rT}$), so

$$F = S\, e^{rT}$$

At $K = F$ the call and put have the same price, since put-call parity gives $C - P = S - K e^{-rT} = 0$ (Session 8). Below $F$ the put is OTM, and at or above $F$ the call is OTM. $F$ is used only for this split. The implied volatility itself (section 1.3) uses $S$.

**Assumption: no dividends.** With a dividend yield $q$ the forward is $F = S e^{(r - q)T}$. We set $q = 0$. With $q$ around 1.3% a year, $F$ is about 3 points too high for the 16 April expiry and about 7 points too high for 15 May, which moves the put/call split by at most one strike.

In [47]:
def forward_from_spot(S, T, r):
    """Forward price from the spot index level with no dividends:  F = S * e^{rT}. Used to split puts and calls."""
    return S * np.exp(r * T)

### 1.3 Black-Scholes implied volatility
Following the course reference `implied_volatility.ipynb`, the option is priced with the Black-Scholes formula from Session 5, which needs only $S$, $K$, $r$, $\sigma$ and $T$:

$$C = S\,\Phi(d_1) - K e^{-rT}\,\Phi(d_2), \qquad P = K e^{-rT}\,\Phi(-d_2) - S\,\Phi(-d_1)$$

$$d_1 = \frac{\ln(S/K) + \left(r + \frac{\sigma^2}{2}\right) T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

The reference covers calls. The put formula follows from put-call parity, $C - P = S - K e^{-rT}$.

**Implied volatility:** solve

$$\text{price} - \text{BlackScholes}(S, K, r, \sigma_K, T) = 0$$

for $\sigma_K$ with Brent's root-finding method on $\sigma \in [10^{-6},\ 5]$, as in the reference.

A solution only exists when the price is inside the no-arbitrage bounds. Otherwise the result is `NaN`:
- call: $\max(S - K e^{-rT}, 0) < C < S$
- put: $\max(K e^{-rT} - S, 0) < P < K e^{-rT}$

**Link to the Black model.** Substituting $S = e^{-rT} F$ turns this into the Black formula $C = e^{-rT}[F\Phi(d_1) - K\Phi(d_2)]$ with $d_1 = [\ln(F/K) + \sigma^2 T/2]/(\sigma\sqrt{T})$. With $F = S e^{rT}$ the two give exactly the same implied volatilities.

In [48]:
def implied_vol(S, K, r, price, T, option_type="call", lo=1e-6, hi=5.0):
    """
    Black-Scholes implied volatility (reference: implied_volatility.ipynb):
    the sigma at which the Black-Scholes price equals the market price.
    option_type: "call" or "put". Returns NaN when the price is outside the no-arbitrage bounds.
    """
    discounted_K = K * np.exp(-r * T)
    if option_type == "call":
        lower, upper = max(S - discounted_K, 0.0), S
    else:
        lower, upper = max(discounted_K - S, 0.0), discounted_K
    if not np.isfinite(price) or price <= lower or price >= upper:
        return np.nan

    def black_scholes(sigma):
        d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        if option_type == "call":
            return S * norm.cdf(d1) - discounted_K * norm.cdf(d2)
        return discounted_K * norm.cdf(-d2) - S * norm.cdf(-d1)

    return brentq(lambda sigma: price - black_scholes(sigma), lo, hi)

### 1.4 `iv_smile`: putting it together
For a given `time_snapshot`:
1. Read the call and put bid/ask at that minute. Treat `-1` as missing and drop strikes with a zero bid. Mid $= (\text{bid} + \text{ask})/2$.
2. $T$ from `time_to_expiry` (section 1.1), using that row's `EXPIRY_DATE`.
3. $S$ = SPX level in `ASSET` at that minute, or $\text{ES mid} \times e^{-rT}$ between 6 April 20:15 and 7 April 09:30 (section 1.2). Then $F = S e^{rT}$ from `forward_from_spot`.
4. **Choose the OTM option at each strike:** put if $K < F$, call if $K \ge F$. Then solve for $\sigma_K$ with `implied_vol(S, K, r, price, T)` (section 1.3).

**Why exactly one option per strike.** At a given minute there is one $F$, so every strike falls on one side of it. A put and a call at the same strike are never both OTM or both ITM: if $K < F$ the put is OTM and the call is ITM, and if $K > F$ it is the other way round. The ITM option at each strike is never used. Each minute is classified on its own, so when the market moves between days, $F$ moves and the strike where the smile switches from puts to calls moves with it.

Returns the smile as a DataFrame (one row per strike, including `OPTION_TYPE` and `K_OVER_F` so the OTM choice can be checked) and a dict with $S$, where it came from, $F$, $T$ and the expiry.

In [49]:
def iv_smile(time_snapshot, r=RISK_FREE_RATE):
    """
    Implied volatility smile at one minute. Orchestrates time_to_expiry, forward_from_spot
    and implied_vol. Returns (smile DataFrame, info dict).
    """
    t = pd.Timestamp(time_snapshot)
    strikes = [c for c in OPTION_CALL_BID.columns if c not in ("TRADING_HOUR", "EXPIRY_DATE")]
    K = np.array(strikes, dtype=float)

    # 1. Mid prices at this minute; -1 = no quote, and a zero bid has no reliable mid
    def quotes(df):
        q = df.loc[t, strikes].astype(float).to_numpy()
        return np.where(q < 0, np.nan, q)

    call_bid, call_ask = quotes(OPTION_CALL_BID), quotes(OPTION_CALL_ASK)
    put_bid, put_ask = quotes(OPTION_PUT_BID), quotes(OPTION_PUT_ASK)
    C = np.where(call_bid > 0, (call_bid + call_ask) / 2, np.nan)
    P = np.where(put_bid > 0, (put_bid + put_ask) / 2, np.nan)

    # 2. Time to expiry
    expiry_date = OPTION_CALL_BID.loc[t, "EXPIRY_DATE"]
    T = time_to_expiry(t, expiry_date)

    # 3. Underlying price S: the SPX level at this minute.
    #    6 Apr 20:15 to 7 Apr 09:30: the overnight SPX value is a bad vendor estimate,
    #    so use the ES mid as the forward and discount it to an equivalent spot level.
    if pd.Timestamp("2025-04-06 20:15") <= t < pd.Timestamp("2025-04-07 09:30"):
        es_mid = (ASSET.loc[t, "ES_BID"] + ASSET.loc[t, "ES_ASK"]) / 2
        S, S_source = es_mid * np.exp(-r * T), "ES mid x e^(-rT)"
    else:
        S, S_source = ASSET.loc[t, "SPX"], "SPX"
    F = forward_from_spot(S, T, r)  # at-the-money point used to split puts and calls

    # 4. Implied vol per strike from the OTM option: put if K < F, call if K >= F
    rows = []
    for k, c, p in zip(K, C, P):
        option_type = "put" if k < F else "call"
        price = p if option_type == "put" else c
        rows.append({
            "STRIKE": k,
            "K_OVER_F": k / F,
            "OPTION_TYPE": option_type,
            "MID_PRICE": price,
            "IMPLIED_VOL": implied_vol(S, k, r, price, T, option_type),
        })
    smile = pd.DataFrame(rows).dropna(subset=["IMPLIED_VOL"]).reset_index(drop=True)

    info = {
        "TIME": t, "EXPIRY_DATE": expiry_date, "T_YEARS": T, "T_DAYS": T * 365,
        "R": r, "S": S, "S_SOURCE": S_source, "F": F,
    }
    return smile, info

### 1.5 Daily smiles
One smile per calendar day in the sample, 1 to 11 April, giving 10 curves. Each day uses the minute closest to the 16:00 ET close. Days without a regular session use their closest available minute: Sunday 6 April uses the 20:15 open of the overnight session, and 11 April uses its last minute, 09:15.

Colour runs in date order from blue (earliest) through purple and pink to orange-red (latest). Solid lines are the 16 April expiry and dashed lines are the 15 May expiry, because the option chain rolls to the May expiry from 6 April. The 6 April curve uses the ES mid price for $F$ (see the known data issue in section 1.2).

The table under the code lists $S$, $F$ and $T$ for each snapshot. The check below it confirms that every put used has $K < F$ and every call has $K \ge F$, and counts how many puts and calls each smile uses.

In [ ]:
# One snapshot per calendar day: the minute closest to 16:00 ET
minutes = OPTION_CALL_BID.index.to_series()
target = minutes.dt.normalize() + pd.Timedelta(hours=16)
daily_snapshots = (minutes - target).abs().groupby(minutes.dt.date).idxmin().tolist()

# Event groups (ET). The 2 Apr 16:00 snapshot is the announcement itself, so it is "Actual Event"
EVENT_TIME = pd.Timestamp("2025-04-02 16:00")
POST_START = pd.Timestamp("2025-04-02 20:15")
def event_group(t):
    if t < EVENT_TIME:
        return "Pre Event"
    if t < POST_START:
        return "Actual Event"
    return "Post Event"

smiles, infos = [], []
for time_snapshot in daily_snapshots:
    smile, info = iv_smile(time_snapshot)
    group = event_group(info["TIME"])
    label = f"{group}: {info['TIME']:%a %d %b %H:%M}"
    smiles.append(smile.assign(SNAPSHOT=label, EXPIRY=f"{info['EXPIRY_DATE']:%d %b} expiry"))
    infos.append({**info, "SNAPSHOT": label, "GROUP": group})

daily_smiles = pd.concat(smiles, ignore_index=True)
labels = [i["SNAPSHOT"] for i in infos]
daily_smiles["SNAPSHOT"] = pd.Categorical(daily_smiles["SNAPSHOT"], categories=labels, ordered=True)
display(pd.DataFrame(infos).set_index("SNAPSHOT")[["GROUP", "EXPIRY_DATE", "T_DAYS", "S", "S_SOURCE", "F"]])

# Check: every put used is below the forward and every call is at or above it (OTM only)
is_put = daily_smiles["OPTION_TYPE"] == "put"
assert (daily_smiles.loc[is_put, "K_OVER_F"] < 1).all() and (daily_smiles.loc[~is_put, "K_OVER_F"] >= 1).all()
display(daily_smiles.groupby(["SNAPSHOT", "OPTION_TYPE"], observed=True).size().unstack())

# ATM point per snapshot: implied vol interpolated at K = F on that day's smile
atm_rows = []
for info in infos:
    curve = daily_smiles[daily_smiles["SNAPSHOT"] == info["SNAPSHOT"]].dropna(subset=["IMPLIED_VOL"]).sort_values("STRIKE")
    atm_rows.append({
        "SNAPSHOT": info["SNAPSHOT"],
        "EXPIRY": f"{info['EXPIRY_DATE']:%d %b} expiry",
        "STRIKE": info["F"],
        "IMPLIED_VOL": np.interp(info["F"], curve["STRIKE"], curve["IMPLIED_VOL"]),
    })
atm_points = pd.DataFrame(atm_rows)
atm_points["SNAPSHOT"] = pd.Categorical(atm_points["SNAPSHOT"], categories=labels, ordered=True)
display(atm_points.set_index("SNAPSHOT").style.format({"STRIKE": "{:,.0f}", "IMPLIED_VOL": "{:.1%}"}))

# One colour family per event group, lighter to darker within each group over time
GROUP_CMAPS = {"Pre Event": "Blues", "Actual Event": "Greens", "Post Event": "Reds"}
colours = {}
for group, cmap in GROUP_CMAPS.items():
    group_labels = [i["SNAPSHOT"] for i in infos if i["GROUP"] == group]
    shades = np.linspace(0.4, 0.95, len(group_labels)) if len(group_labels) > 1 else [0.75]
    for label, shade in zip(group_labels, shades):
        colours[label] = matplotlib.colors.to_hex(matplotlib.colormaps[cmap](shade))

(
    ggplot(daily_smiles, aes("STRIKE", "IMPLIED_VOL", color="SNAPSHOT", linetype="EXPIRY"))
    + geom_line(size=1)
    + geom_point(atm_points, aes("STRIKE", "IMPLIED_VOL", fill="SNAPSHOT"), inherit_aes=False,
                 color="white", size=3.5, stroke=0.8, show_legend=False)
    + scale_fill_manual(values=colours)
    + scale_color_manual(values=colours)
    + scale_linetype_manual(values={"16 Apr expiry": "solid", "15 May expiry": "dashed"})
    + scale_y_continuous(labels=percent_format())
    + labs(
        title="SPX implied volatility smile, daily",
        subtitle=f"Black-Scholes implied vol, minute closest to 16:00 ET each day, r = {RISK_FREE_RATE:.2%}, S = SPX (ES mid on 6 Apr)\nDots mark at-the-money (K = F)",
        x="Strike", y="Implied volatility", color="Snapshot (ET)", linetype="",
    )
    + theme_minimal()
    + theme(figure_size=(9, 5), panel_grid_minor=element_blank())
)

## 2. Plot Volatility Surface

## 3. Stochastic volatility (SABR)

**Question.** Section 1 shows the smile getting higher and steeper as the tariff news broke. Here we ask whether a simple stochastic volatility model, SABR (Session 6), can describe those smiles, and what its parameters say about how the market priced the event.

**Why SABR.** Black-Scholes assumes one constant volatility, so it cannot produce a smile at all. SABR lets the volatility itself move randomly and be correlated with the price:

$$dF_t = \alpha_t F_t^{\beta}\, dW^F_t, \qquad d\alpha_t = \nu\, \alpha_t\, dW^\alpha_t, \qquad dW^F_t\, dW^\alpha_t = \rho\, dt$$

Each parameter controls one feature of the smile:

| Parameter | What it controls | What to look for during the event |
|---|---|---|
| $\alpha$ | Overall **level** of volatility (roughly the at-the-money vol, scaled by $F^{1-\beta}$) | Rises with fear, falls after the pause |
| $\rho$ | Correlation between price and volatility. Negative $\rho$ means volatility rises when the market falls, which makes low-strike puts dearer: the **skew** | More negative = more demand for crash protection |
| $\nu$ | Volatility of volatility. Controls how much both wings lift: the **curvature** | Higher = market paying more for large moves either way |
| $\beta$ | How volatility scales with the price level | Fixed, not calibrated (section 3.2) |

The work follows the course reference `calibration (full).ipynb`.

### 3.1 Hagan's SABR implied volatility
SABR has no closed-form option price, but Hagan et al. (2002) derived an approximation for the Black implied volatility it produces (Session 6, slide 18):

$$\sigma_{\text{SABR}}(F, K) = \frac{\alpha}{(FK)^{(1-\beta)/2}\left\{1 + \frac{(1-\beta)^2}{24}\log^2\frac{F}{K} + \frac{(1-\beta)^4}{1920}\log^4\frac{F}{K}\right\}} \cdot \frac{z}{x(z)} \cdot \left\{1 + \left[\frac{(1-\beta)^2}{24}\frac{\alpha^2}{(FK)^{1-\beta}} + \frac{1}{4}\frac{\rho\beta\nu\alpha}{(FK)^{(1-\beta)/2}} + \frac{2-3\rho^2}{24}\nu^2\right]T\right\}$$

$$z = \frac{\nu}{\alpha}(FK)^{(1-\beta)/2}\log\frac{F}{K}, \qquad x(z) = \log\left[\frac{\sqrt{1-2\rho z+z^2}+z-\rho}{1-\rho}\right]$$

At $K = F$ the ratio $z/x(z)$ tends to 1, so the code uses the simpler at-the-money form there.

**Link to Section 1.** $\sigma_{\text{SABR}}$ is a Black (forward-based) volatility. Section 1.3 shows that, with $F = S e^{rT}$, the Black and Black-Scholes implied vols are identical, so SABR vols can be compared directly with the market vols from `iv_smile`.

The function below is copied unchanged from the course reference.

In [ ]:
def SABR(F, K, T, alpha, beta, rho, nu):
    """Hagan (2002) SABR implied Black volatility. Copied from the course reference calibration (full).ipynb."""
    X = K
    # if K is at-the-money-forward
    if abs(F - K) < 1e-12:
        numer1 = (((1 - beta)**2)/24)*alpha*alpha/(F**(2 - 2*beta))
        numer2 = 0.25*rho*beta*nu*alpha/(F**(1 - beta))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        VolAtm = alpha*(1 + (numer1 + numer2 + numer3)*T)/(F**(1-beta))
        sabrsigma = VolAtm
    else:
        z = (nu/alpha)*((F*X)**(0.5*(1-beta)))*np.log(F/X)
        zhi = np.log((((1 - 2*rho*z + z*z)**0.5) + z - rho)/(1 - rho))
        numer1 = (((1 - beta)**2)/24)*((alpha*alpha)/((F*X)**(1 - beta)))
        numer2 = 0.25*rho*beta*nu*alpha/((F*X)**((1 - beta)/2))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        numer = alpha*(1 + (numer1 + numer2 + numer3)*T)*z
        denom1 = ((1 - beta)**2/24)*(np.log(F/X))**2
        denom2 = (((1 - beta)**4)/1920)*((np.log(F/X))**4)
        denom = ((F*X)**((1 - beta)/2))*(1 + denom1 + denom2)*zhi
        sabrsigma = numer/denom

    return sabrsigma

### 3.2 Calibration
**Calibration** means choosing $\alpha$, $\rho$ and $\nu$ so that SABR's smile matches the market smile. For each strike $K_i$ in a snapshot, the error is

$$\epsilon_i = \sigma_{\text{Mkt}}(K_i) - \sigma_{\text{SABR}}(F, K_i, T, \alpha, \beta, \rho, \nu)$$

and we find

$$\min_{\alpha,\ \rho,\ \nu} \sum_{i=1}^{n} \epsilon_i^2$$

with `scipy.optimize.least_squares`, as in the course reference.

**Assumptions and choices**
- **Market vols, $F$ and $T$:** taken straight from `iv_smile` (Section 1.4), so SABR is fitted to exactly the smiles plotted in Section 1.5.
- **$\beta = 0.7$, fixed.** This is the value the course reference code uses. $\beta$ and $\rho$ both tilt the smile, so fitting both at once is unstable. Section 3.6 shows the fit is almost the same for $\beta$ = 0.5, 0.7 and 1.
- **Every strike counts equally**, as in the course reference.
- **Options priced under \$0.50 are left out of the fit.** Far out-of-the-money options trade at a few ticks (\$0.10 to \$0.50), where the minimum price step and the bid-ask spread decide the price, not volatility. Their implied vols jump around and pull the fit away from the rest of the smile. They are still shown, in grey, on the charts in Section 3.4.
- **Bounds:** $\alpha > 0$, $-1 < \rho < 1$, $\nu > 0$, so the formula stays defined.
- **Starting point:** at the money, $\sigma_{\text{SABR}} \approx \alpha / F^{1-\beta}$, so we start from $\alpha_0 = \sigma_{\text{ATM}} \cdot F^{1-\beta}$, $\rho_0 = -0.5$, $\nu_0 = 1$.

**One change from the course code.** The course's `sabrcalibration` returns the *sum* of squared errors. `least_squares` already squares and sums what it is given, so here `sabr_residuals` returns one error per strike. The minimum is the same, and this is the form `least_squares` expects.

**Fit quality** is reported as the root-mean-square error (RMSE) across strikes, in volatility points: an RMSE of 0.005 means SABR misses the market by about 0.5 vol points on average.

In [ ]:
from scipy.optimize import least_squares

SABR_BETA = 0.7       # fixed, as in the course reference calibration (full).ipynb
MIN_MID_PRICE = 0.50  # options cheaper than this are left out of the fit (tick-size noise)


def sabr_residuals(x, strikes, vols, F, T, beta):
    """One error per strike: market vol minus SABR vol. least_squares squares and sums these itself."""
    alpha, rho, nu = x
    return np.array([vol - SABR(F, K, T, alpha, beta, rho, nu) for K, vol in zip(strikes, vols)])


def calibrate_sabr(smile, info, beta=SABR_BETA, min_price=MIN_MID_PRICE):
    """
    Fit alpha, rho and nu (beta fixed) to one market smile from iv_smile.
    Returns (params dict, smile DataFrame with SABR_VOL, ERROR and USED_IN_FIT columns).
    """
    F, T = info["F"], info["T_YEARS"]
    smile = smile.sort_values("STRIKE").reset_index(drop=True)
    smile["USED_IN_FIT"] = smile["MID_PRICE"] >= min_price
    used = smile[smile["USED_IN_FIT"]]
    strikes, vols = used["STRIKE"].to_numpy(), used["IMPLIED_VOL"].to_numpy()

    # Starting point: at the money, SABR vol is about alpha / F^(1-beta)
    atm_vol = np.interp(F, strikes, vols)
    x0 = [atm_vol * F**(1 - beta), -0.5, 1.0]
    bounds = ([1e-6, -0.999, 1e-6], [np.inf, 0.999, np.inf])

    res = least_squares(sabr_residuals, x0, bounds=bounds, args=(strikes, vols, F, T, beta))
    alpha, rho, nu = res.x

    smile["SABR_VOL"] = [SABR(F, K, T, alpha, beta, rho, nu) for K in smile["STRIKE"]]
    smile["ERROR"] = smile["IMPLIED_VOL"] - smile["SABR_VOL"]
    errors = smile.loc[smile["USED_IN_FIT"], "ERROR"]

    params = {
        "ALPHA": alpha, "BETA": beta, "RHO": rho, "NU": nu,
        "MKT_ATM_VOL": atm_vol,
        "SABR_ATM_VOL": SABR(F, F, T, alpha, beta, rho, nu),
        "RMSE": np.sqrt(np.mean(errors**2)),
        "MAX_ABS_ERROR": errors.abs().max(),
        "N_STRIKES": len(errors),
        "CONVERGED": res.success,
    }
    return params, smile

### 3.3 Snapshots and calibrated parameters
SABR is calibrated at 16 snapshots:
- **The 10 daily snapshots from Section 1.5** (minute closest to 16:00 ET), so the parameters line up with the smiles in Figure 1.
- **6 event moments**, chosen from the events file and the SPX path in `ASSET`:

| Snapshot (ET) | Why |
|---|---|
| Wed 02 Apr 20:20 | First overnight minute after the 16:07 Liberation Day announcement |
| Mon 07 Apr 10:05 | Just before the false "90-day pause" report |
| Mon 07 Apr 10:18 | Peak of the rally on the false report (SPX about 5,165, up from about 4,965) |
| Mon 07 Apr 10:30 | After the White House denial (SPX back near 5,000) |
| Wed 09 Apr 13:15 | Just before the real 90-day pause (13:18) |
| Wed 09 Apr 13:30 | 12 minutes after the pause (SPX up about 4%) |

**Note on the events file.** It times the false report at 09:13 and the denial at 09:35. In `ASSET`, the jump happens at about 10:12 and reverses by about 10:25, so the snapshots follow the market data.

In [ ]:
EVENT_SNAPSHOTS = {
    "2025-04-02 20:20": "Overnight after announcement",
    "2025-04-07 10:05": "Before false pause report",
    "2025-04-07 10:18": "Peak of false pause rally",
    "2025-04-07 10:30": "After White House denial",
    "2025-04-09 13:15": "Before 90-day pause",
    "2025-04-09 13:30": "After 90-day pause",
}
sabr_snapshots = {pd.Timestamp(t): "Daily close" for t in daily_snapshots}
sabr_snapshots.update({pd.Timestamp(t): note for t, note in EVENT_SNAPSHOTS.items()})

param_rows, fit_frames = [], []
for t, note in sorted(sabr_snapshots.items()):
    smile, info = iv_smile(t)
    params, fitted = calibrate_sabr(smile, info)
    label = f"{t:%a %d %b %H:%M}"
    param_rows.append({
        "SNAPSHOT": label, "TIME": t, "TYPE": note,
        "EXPIRY": f"{info['EXPIRY_DATE']:%d %b} expiry", "T_DAYS": info["T_DAYS"], "F": info["F"],
        **params,
    })
    fit_frames.append(fitted.assign(SNAPSHOT=label, TYPE=note, EXPIRY=f"{info['EXPIRY_DATE']:%d %b} expiry"))

SABR_PARAMS = pd.DataFrame(param_rows)
SABR_FITS = pd.concat(fit_frames, ignore_index=True)

# Keep snapshots in time order on every chart
snapshot_order = SABR_PARAMS["SNAPSHOT"].tolist()
SABR_PARAMS["SNAPSHOT"] = pd.Categorical(SABR_PARAMS["SNAPSHOT"], categories=snapshot_order, ordered=True)
SABR_FITS["SNAPSHOT"] = pd.Categorical(SABR_FITS["SNAPSHOT"], categories=snapshot_order, ordered=True)

assert SABR_PARAMS["CONVERGED"].all()
display(
    SABR_PARAMS.set_index("SNAPSHOT")[
        ["TYPE", "EXPIRY", "T_DAYS", "F", "ALPHA", "RHO", "NU", "MKT_ATM_VOL", "SABR_ATM_VOL", "RMSE", "MAX_ABS_ERROR", "N_STRIKES"]
    ].style.format({
        "T_DAYS": "{:.1f}", "F": "{:,.0f}", "ALPHA": "{:.3f}", "RHO": "{:.3f}", "NU": "{:.2f}",
        "MKT_ATM_VOL": "{:.1%}", "SABR_ATM_VOL": "{:.1%}", "RMSE": "{:.4f}", "MAX_ABS_ERROR": "{:.4f}",
    })
)

### 3.4 How well does SABR fit?
Each panel is one snapshot. The x-axis is moneyness $K/F$, so snapshots with different forwards line up (left of 1: puts, right of 1: calls). Coloured dots are the market vols used in the fit, grey dots are the options under \$0.50 left out of the fit, and the line is the calibrated SABR smile.

In [ ]:
EXPIRY_COLOURS = {"16 Apr expiry": "#2b6cb0", "15 May expiry": "#c05621"}

def plot_sabr_fits(fits, title):
    used, dropped = fits[fits["USED_IN_FIT"]], fits[~fits["USED_IN_FIT"]]
    return (
        ggplot(fits, aes("K_OVER_F"))
        + geom_vline(xintercept=1, linetype="dotted", color="grey")
        + geom_point(data=dropped, mapping=aes(y="IMPLIED_VOL"), color="#b0b0b0", size=1.2)
        + geom_point(data=used, mapping=aes(y="IMPLIED_VOL", color="EXPIRY"), size=1.2)
        + geom_line(aes(y="SABR_VOL"), color="black", size=0.8)
        + facet_wrap("SNAPSHOT", ncol=4, scales="free")
        + scale_color_manual(values=EXPIRY_COLOURS)
        + scale_y_continuous(labels=percent_format())
        + labs(
            title=title,
            subtitle=f"Dots: market implied vol (grey: mid under ${MIN_MID_PRICE:.2f}, not fitted). Line: calibrated SABR, beta = {SABR_BETA}",
            x="Moneyness K / F", y="Implied volatility", color="",
        )
        + theme_minimal()
        + theme(figure_size=(12, 8), panel_grid_minor=element_blank(), legend_position="bottom")
    )

display(plot_sabr_fits(SABR_FITS[SABR_FITS["TYPE"] == "Daily close"], "SABR fit to the daily SPX smiles"))
display(plot_sabr_fits(SABR_FITS[SABR_FITS["TYPE"] != "Daily close"], "SABR fit at the event moments"))

### 3.5 Parameters through the event
How $\alpha$, $\rho$, $\nu$ and the fit error moved across all 16 snapshots. Colour marks the expiry: from 6 April the options are for 15 May instead of 16 April, so the two sets of points are not directly comparable (see the reading notes in Section 3.7).

In [ ]:
param_long = SABR_PARAMS.melt(
    id_vars=["SNAPSHOT", "EXPIRY", "TYPE"],
    value_vars=["SABR_ATM_VOL", "RHO", "NU", "RMSE"],
    var_name="PARAMETER", value_name="VALUE",
)
param_long["PARAMETER"] = pd.Categorical(
    param_long["PARAMETER"].map({
        "SABR_ATM_VOL": "ATM vol (level, from alpha)",
        "RHO": "rho (skew)",
        "NU": "nu (vol of vol)",
        "RMSE": "Fit error (RMSE)",
    }),
    categories=["ATM vol (level, from alpha)", "rho (skew)", "nu (vol of vol)", "Fit error (RMSE)"],
    ordered=True,
)
param_long["EVENT_MOMENT"] = np.where(param_long["TYPE"] == "Daily close", "Daily close", "Event moment")

(
    ggplot(param_long, aes("SNAPSHOT", "VALUE", color="EXPIRY", group="EXPIRY"))
    + geom_line(size=0.6, alpha=0.5)
    + geom_point(aes(shape="EVENT_MOMENT"), size=2.5)
    + facet_wrap("PARAMETER", ncol=2, scales="free_y")
    + scale_color_manual(values=EXPIRY_COLOURS)
    + scale_shape_manual(values={"Daily close": "o", "Event moment": "D"})
    + labs(
        title="SABR parameters through the Liberation Day tariff shock",
        subtitle=f"beta fixed at {SABR_BETA}. ATM vol is the SABR vol at K = F, a readable stand-in for alpha.",
        x="", y="", color="", shape="",
    )
    + theme_minimal()
    + theme(figure_size=(12, 8), panel_grid_minor=element_blank(),
            axis_text_x=element_text(rotation=60, ha="right", size=7), legend_position="bottom")
)

### 3.6 Does the choice of $\beta$ matter?
The calibration is repeated with $\beta$ = 0.5, 0.7 and 1 at every snapshot. If the RMSE barely changes, $\beta$ is not driving the results and fixing it at 0.7 is harmless.

In [ ]:
beta_rows = []
for t in sorted(sabr_snapshots):
    smile, info = iv_smile(t)
    for beta in [0.5, 0.7, 1.0]:
        params, _ = calibrate_sabr(smile, info, beta=beta)
        beta_rows.append({"SNAPSHOT": f"{t:%a %d %b %H:%M}", "BETA": beta, "RMSE": params["RMSE"], "RHO": params["RHO"]})

beta_check = pd.DataFrame(beta_rows).pivot(index="SNAPSHOT", columns="BETA", values=["RMSE", "RHO"])
beta_check = beta_check.reindex(snapshot_order)
display(beta_check.style.format("{:.4f}"))

### 3.7 Findings
Numbers below are from the table in Section 3.3 ($\beta$ = 0.7). ATM vol means the SABR vol at $K = F$, which tracks $\alpha$.

**1. Level ($\alpha$) moved with every event.** ATM vol was about 19% at the 1 and 2 April closes. It jumped to 25% in the first overnight hours after the announcement, then rose to 28% (3 April close) and 45% (4 April close, after China's retaliation). It stayed around 37–46% from the Sunday 6 April open to 8 April. Just before the 9 April pause it was 39%. Twelve minutes after the pause it was 31%, and 25% at the close. It drifted back up to 33–36% on 10–11 April.

**2. Skew ($\rho$) steepened as fear built.** $\rho$ went from −0.67 before the announcement to −0.74 about four hours after it, then −0.75 and −0.76 on 3 and 4 April. More negative $\rho$ means puts became relatively dearer than calls: the market paid more for crash protection, not just more for volatility in general. On the May expiry, $\rho$ reached its most negative value (−0.85) at 13:15 on 9 April, three minutes before the pause. Within 15 minutes of the pause it was back to −0.76, and −0.73 at the close, the flattest skew of the May expiry. It steepened again on 10 and 11 April (−0.79, −0.83) as the China escalation remained.

**3. Vol of vol ($\nu$) is driven mostly by maturity.** $\nu$ is 3.3–4.0 on the 16 April options and 2.3–2.9 on the 15 May options. Short-dated smiles are naturally more curved, so this gap is mainly the change of expiry, not the event. Within each expiry, $\nu$ peaked on 4 April (4.0), and on the May expiry it rose from 2.34 just before the pause to 2.89 at the 9 April close. After the pause, the market paid more for large moves in *both* directions, not just down.

**4. The false pause report is a natural experiment.** On 7 April, ATM vol fell from 38.5% (10:05) to 29.5% at the peak of the rally on the false report (10:18), then went straight back to 38.1% after the White House denial (10:30). $\rho$ moved −0.79 → −0.75 → −0.77. Option prices treated the rumour as real for about 15 minutes and then undid it completely. The real pause on 9 April gave a similar first move (39% → 31% in 15 minutes) but kept going (25% by the close): the market separated a rumour from a policy change within minutes.

**5. Fit quality.** SABR fits the 15 May smiles closely (RMSE mostly 0.1–0.6 vol points) and the 16 April smiles less well (0.6–1.0 points). The worst fits are:
- 1–2 April: SABR's ATM vol is 1.1–1.3 points below the market's, because the two-week smile bends more sharply near the money than SABR can produce.
- 13:30 on 9 April: max error of 3.9 points, while the market was repricing within minutes and quotes were inconsistent across strikes.

The choice of $\beta$ makes almost no difference (Section 3.6: RMSE changes by less than 0.1 vol points between $\beta$ = 0.5 and 1).

**Verdict.** With three parameters, SABR describes each snapshot's smile to within about one vol point, and the parameters give a clear picture of each phase: the level and skew rose together as the shock spread, the skew was steepest just before the pause, and level and skew both dropped within minutes of it. But SABR assumes $\rho$ and $\nu$ are constant, and here they had to be recalibrated at every snapshot, sometimes moving sharply within 15 minutes. So SABR is a good *description* of each smile, but not a model of how the smile moved through the event.

**Reading notes and limitations**
- From 6 April the options are for 15 May (about 38 days) instead of 16 April (about 13 days). $\alpha$ and $\nu$ both depend on time to expiry, so the jump between 4 and 6 April mixes the event with the change of expiry. Comparisons within one expiry are the reliable ones.
- On several 15 May snapshots (6 April 20:15, and the 7, 9 and 10 April closes) the put and call vols do not quite meet at $K = F$ (a small step at $K/F = 1$ in Section 3.4). This suggests $F$ is slightly off. Two likely causes: the no-dividend assumption ($F = S e^{rT}$ ignores a dividend yield of about 1.3%), and on 6 April the ES contract, which expires in June and so prices a June forward, not a 15 May one. A forward backed out from put-call parity, $C - P = e^{-rT}(F - K)$, would remove the step. SABR smooths across it, so the effect on the parameters is small.
- Each snapshot is one minute. Single-minute quotes can be noisy, especially in fast markets like 13:30 on 9 April.

## 4. Risk-neutral density and volatility index comparison